In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_PATH = "/content/drive/MyDrive/acllmdb"
!ls /content/drive/MyDrive/acllmdb


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
imdbEr.txt  imdb.vocab	README	test  train


In [ ]:
!ls /content/drive/MyDrive/acllmdb
!ls /content/acllmdb/train
!ls /content/acllmdb/test
!ls /content/acllmdb/train/pos | wc -l
!ls /content/acllmdb/train/neg | wc -l
!ls /content/acllmdb/test/pos | wc -l
!ls /content/acllmdb/test/neg | wc -l

imdbEr.txt  imdb.vocab	README	test  train
ls: cannot access '/content/acllmdb/train': No such file or directory
ls: cannot access '/content/acllmdb/test': No such file or directory
ls: cannot access '/content/acllmdb/train/pos': No such file or directory
0
ls: cannot access '/content/acllmdb/train/neg': No such file or directory
0
ls: cannot access '/content/acllmdb/test/pos': No such file or directory
0
ls: cannot access '/content/acllmdb/test/neg': No such file or directory
0


In [ ]:
!pip install -q transformers datasets scikit-learn tqdm pandas

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 2
LR = 3e-5
NUM_CLASSES = 2
MODEL_NAME = "bert-base-uncased"

In [ ]:
dataset = load_dataset("imdb")

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print(train_df.head())
print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
2  If only to avoid making this type of film in t...      0
3  This film was probably inspired by Godard's Ma...      0
4  Oh, brother...after hearing about this ridicul...      0
label
0    12500
1    12500
Name: count, dtype: int64
label
0    12500
1    12500
Name: count, dtype: int64


In [ ]:
train_df_full = train_df.copy().reset_index(drop=True)
test_df_full = test_df.copy().reset_index(drop=True)

print(train_df_full.shape, test_df_full.shape)
print(train_df_full["label"].value_counts())
print(test_df_full["label"].value_counts())

(25000, 2) (25000, 2)
label
0    12500
1    12500
Name: count, dtype: int64
label
0    12500
1    12500
Name: count, dtype: int64


In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist() if isinstance(texts, pd.Series) else texts
        self.labels = labels.tolist() if isinstance(labels, pd.Series) else labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset = ReviewDataset(
    train_df_full["text"],
    train_df_full["label"],
    tokenizer,
    MAX_LEN
)

test_dataset = ReviewDataset(
    test_df_full["text"],
    test_df_full["label"],
    tokenizer,
    MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(len(train_loader), len(test_loader))

1563 1563


In [ ]:
class BertBiLSTMClassifier(nn.Module):
    def __init__(self, num_classes, lstm_hidden_size=128, lstm_layers=1, dropout=0.3, freeze_bert=False):
        super(BertBiLSTMClassifier, self).__init__()

        self.bert = BertModel.from_pretrained(MODEL_NAME)

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.bilstm = nn.LSTM(
            input_size=768,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden_size * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        bert_outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        last_hidden_state = bert_outputs.last_hidden_state
        cls_embedding = last_hidden_state[:, 0, :]
        lstm_input = cls_embedding.unsqueeze(1)

        _, (hidden, _) = self.bilstm(lstm_input)

        forward_hidden = hidden[-2, :, :]
        backward_hidden = hidden[-1, :, :]

        combined_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        x = self.dropout(combined_hidden)
        logits = self.fc(x)

        return logits

In [ ]:
model = BertBiLSTMClassifier(num_classes=NUM_CLASSES, freeze_bert=False).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, eps=1e-8)

print(sum(p.numel() for p in model.parameters()))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


110402306


In [ ]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(data_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(all_labels, all_preds)

    return avg_loss, acc

In [ ]:
def eval_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(all_labels, all_preds)

    return avg_loss, acc, all_preds, all_labels

In [ ]:
best_val_acc = 0.0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_preds, val_labels = eval_model(model, test_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_bert_bilstm_imdb.pt")
        print("Best model saved.")


Epoch 1/2


100%|██████████| 1563/1563 [03:41<00:00,  7.05it/s]


Train Loss: 0.3360 | Train Acc: 0.8507
Val Loss:   0.2698 | Val Acc:   0.8847
Best model saved.

Epoch 2/2


100%|██████████| 1563/1563 [03:40<00:00,  7.08it/s]

Train Loss: 0.1936 | Train Acc: 0.9240
Val Loss:   0.3086 | Val Acc:   0.8805


In [ ]:
print(classification_report(val_labels, val_preds, digits=4))

              precision    recall  f1-score   support

           0     0.8746    0.8883    0.8814     12500
           1     0.8865    0.8726    0.8795     12500

    accuracy                         0.8805     25000
   macro avg     0.8806    0.8805    0.8805     25000
weighted avg     0.8806    0.8805    0.8805     25000



In [ ]:
def predict_reviews(model, texts, tokenizer, device, max_len=128):
    model.eval()
    predictions = []

    with torch.no_grad():
        for text in texts:
            encoding = tokenizer(
                text,
                add_special_tokens=True,
                max_length=max_len,
                padding="max_length",
                truncation=True,
                return_attention_mask=True,
                return_tensors="pt"
            )

            input_ids = encoding["input_ids"].to(device)
            attention_mask = encoding["attention_mask"].to(device)

            outputs = model(input_ids, attention_mask)
            pred = torch.argmax(outputs, dim=1).item()
            predictions.append(pred)

    return predictions

In [ ]:
sample_reviews = [
    "This movie was absolutely amazing and beautifully acted.",
    "The acting was horrible and the plot was boring.",
    "It had some good moments but overall felt weak."
]

preds = predict_reviews(model, sample_reviews, tokenizer, device, MAX_LEN)
print(preds)

[1, 0, 0]


In [ ]:
def overall_polarity_binary(predictions):
    counts = Counter(predictions)

    positive = counts.get(1, 0)
    negative = counts.get(0, 0)

    if positive >= 1.2 * negative:
        return "positive"
    elif negative >= 1.2 * positive:
        return "negative"
    else:
        return "neutral"

overall = overall_polarity_binary(preds)
print("Overall polarity:", overall)

Overall polarity: negative


In [ ]:
label_map = {0: "negative", 1: "positive"}

for text, pred in zip(sample_reviews, preds):
    print(label_map[pred], "->", text)

positive -> This movie was absolutely amazing and beautifully acted.
negative -> The acting was horrible and the plot was boring.
negative -> It had some good moments but overall felt weak.
